# GPT 시리즈와 스케일링 법칙 실습 - 추가 실습 코드: BPE 토큰화 병합 과정

- Tutorial ID: `mod-gpt-scaling-lab`
- Tutorial: GPT 시리즈와 스케일링 법칙 실습
- Section ID: `practice-bpe-tokenization-python`
- Section: 추가 실습 코드: BPE 토큰화 병합 과정


In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: BPE 토큰화 병합 과정
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from collections import Counter

def get_pairs(tokens):
    return Counter(zip(tokens, tokens[1:]))

def merge_pair(tokens, pair, new_token):
        out=[]; i=0
        while i < len(tokens):
        if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == pair:
            out.append(new_token); i += 2
        else:
            out.append(tokens[i]); i += 1
    return out

text = "low lower newest widest low lower"
words = [list(w) + ['</w>'] for w in text.split()]
print('초기:', words)
for step in range(6):
    pairs = Counter()
        for w in words: pairs.update(get_pairs(w))
    pair, count = pairs.most_common(1)[0]
    new_token = ''.join(pair).replace('</w>','')
    words = [merge_pair(w, pair, new_token) for w in words]
    print(f'{step+1}. merge {pair} count={count} -> {new_token}')
    print('   ', words)
print('\nBPE는 자주 같이 등장하는 byte/character pair를 vocabulary token으로 압축합니다.')